In [125]:
from __future__ import annotations
import logging
import sqlite3
from pathlib import Path

In [126]:
BASE = Path.cwd()

SDM = BASE / 'SDM.db'
SCHEMA = BASE / 'BikeToDrive_RIM - SDM.txt'
LOG = BASE / 'sdm_etl.log'

SOURCES = {
    'accessoire_inkoop': BASE / 'BikeToDrive_4_Accessoire_Inkoop.db',
    'accessoireverkoop': BASE / 'BikeToDrive_1_Accessoireverkoop.db',
    'onderhoud': BASE / 'BikeToDrive_3_Onderhoud.db',
    'fiets_inkoop': BASE / 'BikeToDrive_5_Fiets_Inkoop.db',
    'fietsverkoop': BASE / 'BikeToDrive_2_Fietsverkoop.db',
}

MAPS = [
    ('accessoire_inkoop', 'Leverancier', 'Accessoire_Inkoop_Leverancier', ('leveranciernr',)),
    ('accessoire_inkoop', 'Accessoire', 'Accessoire_Inkoop_Accessoire', ('accessoirenr',)),
    ('accessoire_inkoop', 'Accessoire_Inkoop', 'Accessoire_Inkoop', ('inkoopnr',)),
    ('accessoireverkoop', 'Filiaal', 'Accessoireverkoop_Filiaal', ('filiaalnr',)),
    ('accessoireverkoop', 'Leverancier', 'Accessoireverkoop_Leverancier', ('leveranciernr',)),
    ('accessoireverkoop', 'Klant', 'Accessoireverkoop_Klant', ('klantnr',)),
    ('accessoireverkoop', 'Monteur', 'Accessoireverkoop_Monteur', ('monteurnr',)),
    ('accessoireverkoop', 'Accessoire', 'Accessoireverkoop_Accessoire', ('accessoirenr',)),
    ('accessoireverkoop', 'Accessoire_Verkoop', 'Accessoireverkoop_Accessoire_Verkoop', ('accessoire_verkoopnr',)),
    ('onderhoud', 'Fabrikant', 'Onderhoud_Fabrikant', ('fabrikantnr',)),
    ('onderhoud', 'Filiaal', 'Onderhoud_Filiaal', ('filiaalnr',)),
    ('onderhoud', 'Fiets', 'Onderhoud_Fiets', ('fietsnr',)),
    ('onderhoud', 'Monteur', 'Onderhoud_Monteur', ('monteurnr',)),
    ('onderhoud', 'Onderhoud', 'Onderhoud', ('onderhoudnr',)),
    ('fiets_inkoop', 'Fabrikant', 'Fiets_Inkoop_Fabrikant', ('fabrikantnr',)),
    ('fiets_inkoop', 'Fiets', 'Fiets_Inkoop_Fiets', ('fietsnr',)),
    ('fiets_inkoop', 'Fiets_Inkoop', 'Fiets_Inkoop', ('inkoopnr',)),
    ('fietsverkoop', 'Filiaal', 'Fietsverkoop_Filiaal', ('filiaalnr',)),
    ('fietsverkoop', 'Klant', 'Fietsverkoop_Klant', ('klantnr',)),
    ('fietsverkoop', 'Fabrikant', 'Fietsverkoop_Fabrikant', ('fabrikantnr',)),
    ('fietsverkoop', 'Monteur', 'Fietsverkoop_Monteur', ('monteurnr',)),
    ('fietsverkoop', 'Fiets', 'Fietsverkoop_Fiets', ('fietsnr',)),
    ('fietsverkoop', 'Fiets_Verkoop', 'Fietsverkoop_Fiets_Verkoop', ('fiets_verkoopnr',)),
]

In [127]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler(LOG, encoding='utf-8'), logging.StreamHandler()]
)

In [128]:
def q(name):
    return f'"{name.replace(chr(34), chr(34) * 2)}"'

def cols(conn, table):
    return [r[1] for r in conn.execute(f'PRAGMA table_info({q(table)})')]

def exists(conn, table):
    return conn.execute(
        "SELECT 1 FROM sqlite_master WHERE type='table' AND name=?",
        (table,)
    ).fetchone() is not None

In [129]:
missing = [str(p) for p in [SCHEMA, SDM, *SOURCES.values()] if not p.exists()]
if missing:
    raise FileNotFoundError('Ontbrekende bestanden:\n- ' + '\n- '.join(missing))

In [130]:
sdm = sqlite3.connect(SDM)
sdm.execute('PRAGMA foreign_keys = ON')

if not exists(sdm, MAPS[0][2]):
    logging.info('SDM-schema ontbreekt; opnieuw opbouwen...')
    sdm.executescript(SCHEMA.read_text(encoding='utf-8'))
    sdm.commit()

In [131]:
reset_first = True

if reset_first:
    sdm.execute('PRAGMA foreign_keys = OFF')
    for _, _, table, _ in reversed(MAPS):
        if exists(sdm, table):
            sdm.execute(f'DELETE FROM {q(table)}')
    sdm.commit()
    sdm.execute('PRAGMA foreign_keys = ON')
    logging.info('Reset van SDM is klaar.')

2026-03-19 16:21:45,682 - INFO - Reset van SDM is klaar.


In [132]:
source_conns = {name: sqlite3.connect(path) for name, path in SOURCES.items()}

In [133]:
def sync_table(conn, source_conn, source_table, target_table, keys):
    target_cols = cols(conn, target_table)
    source_cols = cols(source_conn, source_table)

    if target_cols != source_cols:
        raise ValueError(f'{source_table} -> {target_table}: {source_cols} != {target_cols}')

    tmp = f'tmp_{target_table}'
    col_sql = ', '.join(map(q, target_cols))
    placeholders = ', '.join('?' * len(target_cols))
    non_keys = [c for c in target_cols if c not in keys]
    match = ' AND '.join(f'bron.{q(k)} = doel.{q(k)}' for k in keys)

    conn.execute(f'DROP TABLE IF EXISTS {q(tmp)}')
    conn.execute(f'CREATE TEMP TABLE {q(tmp)} AS SELECT {col_sql} FROM {q(target_table)} WHERE 0')

    rows = source_conn.execute(f'SELECT {col_sql} FROM {q(source_table)}').fetchall()
    if rows:
        conn.executemany(f'INSERT INTO {q(tmp)} VALUES ({placeholders})', rows)

    conn.execute(f'''
        INSERT INTO {q(target_table)} ({col_sql})
        SELECT {', '.join(f'bron.{q(c)}' for c in target_cols)}
        FROM {q(tmp)} AS bron
        WHERE NOT EXISTS (
            SELECT 1 FROM {q(target_table)} AS doel WHERE {match}
        )
    ''')

    if non_keys:
        set_sql = ', '.join(
            f'{q(c)} = (SELECT bron.{q(c)} FROM {q(tmp)} AS bron WHERE {match})'
            for c in non_keys
        )
        diff_sql = ' OR '.join(
            f'doel.{q(c)} IS NOT (SELECT bron.{q(c)} FROM {q(tmp)} AS bron WHERE {match})'
            for c in non_keys
        )

        conn.execute(f'''
            UPDATE {q(target_table)} AS doel
            SET {set_sql}
            WHERE EXISTS (SELECT 1 FROM {q(tmp)} AS bron WHERE {match})
              AND ({diff_sql})
        ''')

    conn.execute(f'''
        DELETE FROM {q(target_table)} AS doel
        WHERE NOT EXISTS (
            SELECT 1 FROM {q(tmp)} AS bron WHERE {match}
        )
    ''')

    conn.execute(f'DROP TABLE IF EXISTS {q(tmp)}')
    conn.commit()

In [134]:
for db_name, source_table, target_table, keys in MAPS:
    logging.info('Sync: %s.%s -> %s', db_name, source_table, target_table)
    sync_table(sdm, source_conns[db_name], source_table, target_table, keys)

logging.info('Alle databronnen zijn succesvol naar het SDM gesynchroniseerd.')

2026-03-19 16:21:45,799 - INFO - Sync: accessoire_inkoop.Leverancier -> Accessoire_Inkoop_Leverancier
2026-03-19 16:21:45,810 - INFO - Sync: accessoire_inkoop.Accessoire -> Accessoire_Inkoop_Accessoire
2026-03-19 16:21:45,818 - INFO - Sync: accessoire_inkoop.Accessoire_Inkoop -> Accessoire_Inkoop
2026-03-19 16:21:45,827 - INFO - Sync: accessoireverkoop.Filiaal -> Accessoireverkoop_Filiaal
2026-03-19 16:21:45,835 - INFO - Sync: accessoireverkoop.Leverancier -> Accessoireverkoop_Leverancier
2026-03-19 16:21:45,845 - INFO - Sync: accessoireverkoop.Klant -> Accessoireverkoop_Klant
2026-03-19 16:21:45,854 - INFO - Sync: accessoireverkoop.Monteur -> Accessoireverkoop_Monteur
2026-03-19 16:21:45,863 - INFO - Sync: accessoireverkoop.Accessoire -> Accessoireverkoop_Accessoire
2026-03-19 16:21:45,874 - INFO - Sync: accessoireverkoop.Accessoire_Verkoop -> Accessoireverkoop_Accessoire_Verkoop
2026-03-19 16:21:45,887 - INFO - Sync: onderhoud.Fabrikant -> Onderhoud_Fabrikant
2026-03-19 16:21:45,896 

In [135]:
tables = [
    r[0] for r in sdm.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
    )
]

for table in tables:
    total = sdm.execute(f'SELECT COUNT(*) FROM {q(table)}').fetchone()[0]
    print(f'{table}: {total} rijen')

Accessoire_Inkoop: 50 rijen
Accessoire_Inkoop_Accessoire: 13 rijen
Accessoire_Inkoop_Leverancier: 5 rijen
Accessoireverkoop_Accessoire: 10 rijen
Accessoireverkoop_Accessoire_Verkoop: 100 rijen
Accessoireverkoop_Filiaal: 4 rijen
Accessoireverkoop_Klant: 20 rijen
Accessoireverkoop_Leverancier: 5 rijen
Accessoireverkoop_Monteur: 10 rijen
Fiets_Inkoop: 100 rijen
Fiets_Inkoop_Fabrikant: 10 rijen
Fiets_Inkoop_Fiets: 75 rijen
Fietsverkoop_Fabrikant: 10 rijen
Fietsverkoop_Fiets: 75 rijen
Fietsverkoop_Fiets_Verkoop: 150 rijen
Fietsverkoop_Filiaal: 4 rijen
Fietsverkoop_Klant: 25 rijen
Fietsverkoop_Monteur: 10 rijen
Onderhoud: 50 rijen
Onderhoud_Fabrikant: 11 rijen
Onderhoud_Fiets: 30 rijen
Onderhoud_Filiaal: 5 rijen
Onderhoud_Monteur: 15 rijen


In [136]:
for conn in source_conns.values():
    conn.close()

sdm.close()